# Entregable 2 · Pipeline con tablas Delta y validaciones de calidad

**Escenario.** Una aseguradora recibe un CSV de siniestros con errores típicos de un sistema real: identificadores vacíos, fechas en formatos distintos o imposibles, importes como `N/A` o negativos, estados que no existen. El pipeline debe dejar pasar solo lo fiable, **sin perder nada**: lo que no supera las reglas va a cuarentena con su motivo.

| Capa | Tabla | Qué hace |
|---|---|---|
| Bronce | `bronce_siniestros` | Copia exacta del CSV (todo como texto) + metadatos de ingesta |
| Plata | `plata_siniestros` | Tipos correctos, reglas de calidad aplicadas, **restricciones Delta**, carga idempotente con `MERGE` |
| Cuarentena | `cuarentena_siniestros` | Filas rechazadas, con la lista de reglas que incumplieron |
| Métricas | `calidad_metricas` | Una fila por regla y ejecución: cuántas filas la incumplieron |
| Oro | `oro_resumen_siniestros` | Resumen por tipo y estado para informes |

Las reglas siguen el modelo de las **expectativas** de Lakeflow Declarative Pipelines (laboratorio 09 del curso), con tres acciones posibles:

- **descartar**: la fila no pasa a plata y va a cuarentena.
- **advertir**: la fila pasa, marcada con la advertencia.
- **fallar**: si alguna fila la incumple, el pipeline se detiene.

*Datos: `insurance_claims_raw.csv`, del repositorio de laboratorios DP-750 de Microsoft Learning (licencia MIT).*

In [1]:
import os

EN_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

if EN_DATABRICKS:
    # En Databricks: catálogo y esquema llegan como parámetros (widgets / parámetros del job)
    dbutils.widgets.text('catalogo', 'adb-dp750')
    dbutils.widgets.text('esquema', 'curso09')
    CATALOGO = dbutils.widgets.get('catalogo')
    ESQUEMA = dbutils.widgets.get('esquema')
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{CATALOGO}`.{ESQUEMA}')
    spark.sql(f'CREATE VOLUME IF NOT EXISTS `{CATALOGO}`.{ESQUEMA}.datos_crudos')
    RUTA_DATOS = f'/Volumes/{CATALOGO}/{ESQUEMA}/datos_crudos'
    PREFIJO = f'`{CATALOGO}`.{ESQUEMA}'

    # Si el CSV aún no está en el volumen, se descarga del repositorio
    import urllib.request
    for archivo in ['insurance_claims_raw.csv']:
        destino = f'{RUTA_DATOS}/{archivo}'
        if not os.path.exists(destino):
            urllib.request.urlretrieve('https://raw.githubusercontent.com/MiguelAP30/learning-data-science/main/cursos/09-databricks/datasets/' + archivo, destino)
else:
    # Fuera de Databricks: Spark + Delta Lake en local (ver entorno_local.py)
    from entorno_local import sesion_spark_local
    spark = sesion_spark_local('curso09-pipeline')
    ESQUEMA = 'curso09'
    spark.sql(f'CREATE DATABASE IF NOT EXISTS {ESQUEMA}')
    RUTA_DATOS = '../datasets'
    PREFIJO = ESQUEMA


def tabla(nombre):
    """Nombre completo de una tabla: catálogo.esquema.tabla en Databricks, esquema.tabla en local."""
    return f'{PREFIJO}.{nombre}'


print(f'Entorno: {"Databricks" if EN_DATABRICKS else "local"} · Spark {spark.version} · tablas en {PREFIJO}')

Entorno: local · Spark 4.0.1 · tablas en curso09


## 1. Bronce: ingesta sin transformar

Sin esquema ni inferencia: **todo se lee como texto**. Si se intentara convertir aquí, `not-a-date` o `N/A` romperían la carga o se volverían nulos sin dejar rastro. En bronce se guarda lo que llegó, tal cual.

In [2]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F

ID_EJECUCION = str(uuid.uuid4())[:8]

bronce = (spark.read.option('header', True).csv(f'{RUTA_DATOS}/insurance_claims_raw.csv')
          .withColumn('_fecha_ingesta', F.current_timestamp())
          .withColumn('_archivo_origen', F.col('_metadata.file_name'))
          .withColumn('_id_ejecucion', F.lit(ID_EJECUCION)))
bronce.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable(tabla('bronce_siniestros'))

total = spark.table(tabla('bronce_siniestros')).count()
print(f'Ejecución {ID_EJECUCION} · bronce_siniestros: {total} filas')
spark.table(tabla('bronce_siniestros')).drop('_fecha_ingesta', '_archivo_origen', '_id_ejecucion').show(25, truncate=False)

Ejecución 89543031 · bronce_siniestros: 25 filas


+--------+---------+-----------+----------+-------------+------------+----------+--------------+---------------+----------+
|claim_id|policy_id|customer_id|claim_date|incident_date|claim_amount|claim_type|status        |coverage_amount|agent_code|
+--------+---------+-----------+----------+-------------+------------+----------+--------------+---------------+----------+
|CLM-001 |POL-1001 |CUST-001   |2026-01-05|2026-01-03   |4500.00     |AUTO      |OPEN          |25000.00       |AGT-01    |
|CLM-002 |POL-1002 |CUST-002   |2026-01-07|2026-01-06   |12000.00    |HOME      |PENDING       |150000.00      |AGT-02    |
|CLM-003 |POL-1003 |CUST-003   |2026-01-10|2026-01-09   |250.00      |HEALTH    |CLOSED        |5000.00        |AGT-01    |
|CLM-004 |NULL     |CUST-004   |2026-01-12|2026-01-11   |8900.00     |AUTO      |OPEN          |30000.00       |AGT-03    |
|CLM-005 |POL-1005 |CUST-005   |2026-01-14|2026-01-13   |NULL        |LIFE      |PENDING       |200000.00      |AGT-02    |
|NULL   

## 2. Tipado seguro

`try_to_timestamp` y `try_cast` devuelven **nulo** cuando el valor no se puede convertir, en lugar de lanzar un error. Así, `not-a-date`, `26/02/2026` o `N/A` no detienen el pipeline: quedan como nulos que las reglas detectarán. Se conservan también los valores originales (`*_crudo`) para que la cuarentena muestre qué llegó exactamente.

In [3]:
tipado = spark.table(tabla('bronce_siniestros')).select(
    'claim_id', 'policy_id', 'customer_id',
    F.expr("CAST(try_to_timestamp(claim_date, 'yyyy-MM-dd') AS DATE)").alias('claim_date'),
    F.expr("CAST(try_to_timestamp(incident_date, 'yyyy-MM-dd') AS DATE)").alias('incident_date'),
    F.expr('try_cast(claim_amount AS DECIMAL(12,2))').alias('claim_amount'),
    'claim_type', 'status',
    F.expr('try_cast(coverage_amount AS DECIMAL(12,2))').alias('coverage_amount'),
    'agent_code',
    F.col('claim_date').alias('claim_date_crudo'),
    F.col('incident_date').alias('incident_date_crudo'),
    F.col('claim_amount').alias('claim_amount_crudo'),
)

## 3. Reglas de calidad

Cada regla es una **condición que debe cumplirse**, escrita en SQL. Las que comparan dos columnas solo se evalúan si ambas existen; si falta alguna, ya la detecta otra regla y así la fila no se penaliza dos veces por el mismo problema.

In [4]:
REGLAS = [
    # (nombre,                         condición que debe cumplirse,                                       acción)
    ('claim_id_presente',              'claim_id IS NOT NULL',                                             'descartar'),
    ('customer_id_presente',           'customer_id IS NOT NULL',                                          'descartar'),
    ('fecha_reclamacion_valida',       'claim_date IS NOT NULL',                                           'descartar'),
    ('fecha_incidente_valida',         'incident_date IS NOT NULL',                                        'descartar'),
    ('incidente_antes_de_reclamacion', 'claim_date IS NULL OR incident_date IS NULL OR incident_date <= claim_date', 'descartar'),
    ('importe_valido',                 'claim_amount IS NOT NULL',                                         'descartar'),
    ('importe_no_negativo',            'claim_amount IS NULL OR claim_amount >= 0',                        'descartar'),
    ('poliza_presente',                'policy_id IS NOT NULL',                                            'advertir'),
    ('estado_conocido',                "status IN ('OPEN', 'PENDING', 'CLOSED')",                          'advertir'),
    ('tipo_conocido',                  "claim_type IN ('AUTO', 'HOME', 'HEALTH', 'LIFE')",                 'advertir'),
    ('importe_dentro_de_cobertura',    'claim_amount IS NULL OR claim_amount <= coverage_amount',          'advertir'),
    ('cobertura_positiva',             'coverage_amount > 0',                                              'fallar'),
]


def incumplidas(accion):
    """Array con los nombres de las reglas de esa acción que la fila NO cumple (nulo cuenta como incumplida)."""
    return F.array_compact(F.array(*[
        F.when(~F.coalesce(F.expr(cond), F.lit(False)), F.lit(nombre))
        for nombre, cond, acc in REGLAS if acc == accion
    ]))


evaluado = (tipado
            .withColumn('motivos_descarte', incumplidas('descartar'))
            .withColumn('advertencias', incumplidas('advertir'))
            .withColumn('fallos', incumplidas('fallar')))

evaluado.select('claim_id', 'claim_date_crudo', 'claim_amount_crudo', 'status', 'motivos_descarte', 'advertencias') \
        .filter((F.size('motivos_descarte') > 0) | (F.size('advertencias') > 0)).show(25, truncate=False)

+--------+----------------+------------------+--------------+---------------------------------------------+-----------------+
|claim_id|claim_date_crudo|claim_amount_crudo|status        |motivos_descarte                             |advertencias     |
+--------+----------------+------------------+--------------+---------------------------------------------+-----------------+
|CLM-004 |2026-01-12      |8900.00           |OPEN          |[]                                           |[poliza_presente]|
|CLM-005 |2026-01-14      |NULL              |PENDING       |[importe_valido]                             |[]               |
|NULL    |2026-01-15      |3200.00           |CLOSED        |[claim_id_presente]                          |[]               |
|CLM-007 |not-a-date      |7600.00           |OPEN          |[fecha_reclamacion_valida]                   |[]               |
|CLM-008 |2026-01-18      |N/A               |PENDING       |[importe_valido]                             |[]         

### Métricas por regla

Una fila por regla en cada ejecución, en una tabla Delta que se va acumulando. Es lo que en Lakeflow se ve en el panel de calidad de datos del pipeline, y permite ver si la calidad del origen empeora con el tiempo.

In [5]:
metricas = spark.createDataFrame([
    (ID_EJECUCION, datetime.now(), nombre, accion,
     evaluado.filter(~F.coalesce(F.expr(cond), F.lit(False))).count(), total)
    for nombre, cond, accion in REGLAS
], 'id_ejecucion STRING, fecha TIMESTAMP, regla STRING, accion STRING, filas_incumplen LONG, filas_total LONG') \
    .withColumn('pct_incumplen', F.round(F.col('filas_incumplen') / F.col('filas_total') * 100, 1))

metricas.write.format('delta').mode('append').saveAsTable(tabla('calidad_metricas'))
metricas.select('regla', 'accion', 'filas_incumplen', 'pct_incumplen').orderBy(F.desc('filas_incumplen')).show(truncate=False)

+------------------------------+---------+---------------+-------------+
|regla                         |accion   |filas_incumplen|pct_incumplen|
+------------------------------+---------+---------------+-------------+
|estado_conocido               |advertir |4              |16.0         |
|importe_valido                |descartar|3              |12.0         |
|importe_no_negativo           |descartar|3              |12.0         |
|poliza_presente               |advertir |3              |12.0         |
|fecha_reclamacion_valida      |descartar|2              |8.0          |
|tipo_conocido                 |advertir |1              |4.0          |
|claim_id_presente             |descartar|1              |4.0          |
|fecha_incidente_valida        |descartar|1              |4.0          |
|importe_dentro_de_cobertura   |advertir |0              |0.0          |
|customer_id_presente          |descartar|0              |0.0          |
|cobertura_positiva            |fallar   |0        

### Reglas que detienen el pipeline

Si alguna regla con acción **fallar** se incumple, se lanza una excepción: el resto del cuaderno no se ejecuta, y en un job la tarea queda en rojo y las tareas que dependen de ella no corren.

In [6]:
n_fallos = evaluado.filter(F.size('fallos') > 0).count()
if n_fallos:
    raise ValueError(f'{n_fallos} filas incumplen reglas críticas: se detiene el pipeline')
print('Reglas críticas: OK, ninguna fila las incumple')

Reglas críticas: OK, ninguna fila las incumple


## 4. Cuarentena

Las filas con algún motivo de descarte se guardan con sus valores originales, los motivos y el identificador de ejecución. Nada se pierde: alguien puede revisarlas, corregir el origen y reprocesar.

In [7]:
cuarentena = (evaluado.filter(F.size('motivos_descarte') > 0)
              .select('claim_id', 'policy_id', 'customer_id', 'claim_date_crudo', 'incident_date_crudo',
                      'claim_amount_crudo', 'claim_type', 'status', 'motivos_descarte')
              .withColumn('_id_ejecucion', F.lit(ID_EJECUCION))
              .withColumn('_fecha_cuarentena', F.current_timestamp()))
cuarentena.write.format('delta').mode('append').saveAsTable(tabla('cuarentena_siniestros'))
print(f'A cuarentena: {cuarentena.count()} de {total} filas')
cuarentena.drop('_fecha_cuarentena').show(truncate=False)

A cuarentena: 9 de 25 filas


+--------+---------+-----------+----------------+-------------------+------------------+----------+--------------+---------------------------------------------+-------------+
|claim_id|policy_id|customer_id|claim_date_crudo|incident_date_crudo|claim_amount_crudo|claim_type|status        |motivos_descarte                             |_id_ejecucion|
+--------+---------+-----------+----------------+-------------------+------------------+----------+--------------+---------------------------------------------+-------------+
|CLM-005 |POL-1005 |CUST-005   |2026-01-14      |2026-01-13         |NULL              |LIFE      |PENDING       |[importe_valido]                             |89543031     |
|NULL    |POL-1006 |CUST-006   |2026-01-15      |2026-01-14         |3200.00           |HOME      |CLOSED        |[claim_id_presente]                          |89543031     |
|CLM-007 |POL-1007 |CUST-007   |not-a-date      |2026-01-16         |7600.00           |AUTO      |OPEN          |[fecha_recl

## 5. Plata: tabla Delta con restricciones

La tabla se crea con un esquema explícito y **restricciones**: `NOT NULL` en las columnas clave y dos `CHECK`. Así la propia tabla defiende su calidad. Aunque otro proceso intente escribir saltándose este cuaderno, Delta rechaza la escritura.

In [8]:
PLATA = tabla('plata_siniestros')
spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {PLATA} (
        claim_id         STRING        NOT NULL,
        policy_id        STRING,
        customer_id      STRING        NOT NULL,
        claim_date       DATE          NOT NULL,
        incident_date    DATE          NOT NULL,
        claim_amount     DECIMAL(12,2) NOT NULL,
        claim_type       STRING,
        status           STRING,
        coverage_amount  DECIMAL(12,2),
        agent_code       STRING,
        advertencias     ARRAY<STRING>,
        _id_ejecucion    STRING,
        _fecha_proceso   TIMESTAMP
    ) USING DELTA
''')

RESTRICCIONES = {
    'importe_no_negativo': 'claim_amount >= 0',
    'incidente_antes_de_reclamacion': 'incident_date <= claim_date',
}
existentes = {r['key'] for r in spark.sql(f'SHOW TBLPROPERTIES {PLATA}').collect()}
for nombre, condicion in RESTRICCIONES.items():
    if f'delta.constraints.{nombre}' not in existentes:
        spark.sql(f'ALTER TABLE {PLATA} ADD CONSTRAINT {nombre} CHECK ({condicion})')

spark.sql(f'SHOW TBLPROPERTIES {PLATA}').filter(F.col('key').startswith('delta.constraints')).show(truncate=False)

+------------------------------------------------+---------------------------+
|key                                             |value                      |
+------------------------------------------------+---------------------------+
|delta.constraints.importe_no_negativo           |claim_amount >= 0          |
|delta.constraints.incidente_antes_de_reclamacion|incident_date <= claim_date|
+------------------------------------------------+---------------------------+



### Carga idempotente con `MERGE`

`MERGE` inserta los siniestros nuevos y actualiza los existentes por `claim_id`. Es **idempotente**: si el job se reintenta o se reprocesa el mismo archivo, no se duplican filas.

In [9]:
validos = (evaluado.filter(F.size('motivos_descarte') == 0)
           .select('claim_id', 'policy_id', 'customer_id', 'claim_date', 'incident_date', 'claim_amount',
                   'claim_type', 'status', 'coverage_amount', 'agent_code', 'advertencias')
           .withColumn('_id_ejecucion', F.lit(ID_EJECUCION))
           .withColumn('_fecha_proceso', F.current_timestamp()))
validos.createOrReplaceTempView('lote_valido')

MERGE_SQL = f'''
    MERGE INTO {PLATA} AS destino
    USING lote_valido AS origen
    ON destino.claim_id = origen.claim_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
'''
spark.sql(MERGE_SQL)
print(f'plata_siniestros: {spark.table(PLATA).count()} filas')
spark.table(PLATA).select('claim_id', 'claim_date', 'claim_amount', 'claim_type', 'status', 'advertencias').show(25, truncate=False)

plata_siniestros: 16 filas


+--------+----------+------------+----------+------------+-----------------+
|claim_id|claim_date|claim_amount|claim_type|status      |advertencias     |
+--------+----------+------------+----------+------------+-----------------+
|CLM-001 |2026-01-05|4500.00     |AUTO      |OPEN        |[]               |
|CLM-002 |2026-01-07|12000.00    |HOME      |PENDING     |[]               |
|CLM-003 |2026-01-10|250.00      |HEALTH    |CLOSED      |[]               |
|CLM-004 |2026-01-12|8900.00     |AUTO      |OPEN        |[poliza_presente]|
|CLM-010 |2026-01-22|15000.00    |VEHICLE   |CLOSED      |[tipo_conocido]  |
|CLM-011 |2026-01-25|1800.00     |AUTO      |PENDING     |[]               |
|CLM-012 |2026-01-27|6400.00     |HEALTH    |OPEN        |[]               |
|CLM-013 |2026-01-28|92000.00    |LIFE      |CLOSED      |[]               |
|CLM-015 |2026-02-01|3300.00     |HOME      |OPEN        |[poliza_presente]|
|CLM-016 |2026-02-03|5500.00     |AUTO      |CLOSED      |[]               |

### Las restricciones en acción

Se intenta insertar directamente un siniestro con importe negativo, saltándose las reglas del cuaderno. Delta lo rechaza.

In [10]:
try:
    spark.sql(f'''INSERT INTO {PLATA} VALUES
        ('CLM-999', 'POL-9', 'CUST-9', DATE'2026-03-01', DATE'2026-02-28', -100.00,
         'AUTO', 'OPEN', 1000.00, 'AGT-01', array(), 'prueba', current_timestamp())''')
    print('AVISO: la fila se insertó (no debería)')
except Exception as e:
    import re
    m = re.search(r'\[DELTA_[A-Z_]+\][^\n]*(\n\s*- [^\n]*)*', str(e))
    print('[OK]    Delta rechazó la escritura:')
    print(m.group(0) if m else str(e).splitlines()[0])

[OK]    Delta rechazó la escritura:
[DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint importe_no_negativo (claim_amount >= 0) violated by row with values:
 - claim_amount : -100.00


## 6. Oro: resumen para informes

In [11]:
oro = (spark.table(PLATA)
       .groupBy('claim_type', 'status')
       .agg(F.count('*').alias('siniestros'),
            F.sum('claim_amount').alias('importe_total'),
            F.round(F.avg('claim_amount'), 2).alias('importe_medio'),
            F.round(F.sum('claim_amount') / F.sum('coverage_amount') * 100, 1).alias('pct_sobre_cobertura'),
            F.sum(F.when(F.size('advertencias') > 0, 1).otherwise(0)).alias('con_advertencias'))
       .orderBy('claim_type', 'status'))
oro.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable(tabla('oro_resumen_siniestros'))
spark.table(tabla('oro_resumen_siniestros')).show(truncate=False)

+----------+------------+----------+-------------+-------------+-------------------+----------------+
|claim_type|status      |siniestros|importe_total|importe_medio|pct_sobre_cobertura|con_advertencias|
+----------+------------+----------+-------------+-------------+-------------------+----------------+
|AUTO      |CLOSED      |2         |14300.00     |7150.00      |23.8               |1               |
|AUTO      |OPEN        |2         |13400.00     |6700.00      |24.4               |1               |
|AUTO      |PENDING     |1         |1800.00      |1800.00      |9.0                |0               |
|HEALTH    |CLOSED      |1         |250.00       |250.00       |5.0                |0               |
|HEALTH    |OPEN        |1         |6400.00      |6400.00      |42.7               |0               |
|HEALTH    |PENDING     |1         |780.00       |780.00       |6.5                |0               |
|HOME      |OPEN        |3         |38400.00     |12800.00     |10.7              

## 7. Versionado de datos: historial y viaje en el tiempo

Cada escritura en una tabla Delta crea una **versión**. El historial dice qué operación se hizo, cuándo y con qué efecto. Para comprobar la idempotencia, se **reprocesa el mismo lote**: el `MERGE` vuelve a ejecutarse, crea una versión nueva, pero no inserta ninguna fila.

In [12]:
antes = spark.table(PLATA).count()
spark.sql(MERGE_SQL)  # mismo lote otra vez, como si el job se hubiera reintentado
despues = spark.table(PLATA).count()
print(f'Filas antes del reproceso: {antes} · después: {despues}')

historial = spark.sql(f'DESCRIBE HISTORY {PLATA}')
(historial.select('version', 'operation',
                  F.col('operationMetrics.numTargetRowsInserted').alias('insertadas'),
                  F.col('operationMetrics.numTargetRowsUpdated').alias('actualizadas'))
          .orderBy('version').show(truncate=False))

Filas antes del reproceso: 16 · después: 16


+-------+--------------+----------+------------+
|version|operation     |insertadas|actualizadas|
+-------+--------------+----------+------------+
|0      |CREATE TABLE  |NULL      |NULL        |
|1      |ADD CONSTRAINT|NULL      |NULL        |
|2      |ADD CONSTRAINT|NULL      |NULL        |
|3      |MERGE         |16        |0           |
|4      |MERGE         |0         |16          |
+-------+--------------+----------+------------+



In [13]:
ultima = historial.agg(F.max('version')).first()[0]
version_carga = (historial.filter("operation = 'MERGE'").agg(F.min('version')).first()[0])
print(f'Versión actual: {ultima}')
for v in [0, version_carga]:
    n = spark.sql(f'SELECT COUNT(*) AS n FROM {PLATA} VERSION AS OF {v}').first()['n']
    print(f'  VERSION AS OF {v}: {n} filas')

Versión actual: 4


  VERSION AS OF 0: 0 filas


  VERSION AS OF 3: 16 filas
